In [0]:
# === J9 - ML training setup with proper package installation ===

# Step 1: Install packages
%pip install xgboost==2.0.3 scikit-learn==1.4.0 numpy==1.26.4 -q

# Step 2: Restart Python kernel to apply changes
dbutils.library.restartPython()

In [0]:
# === J9 - Imports (after kernel restart) ===
import sys
import os

repo_root = "/Workspace/Users/salmacharki721@gmail.com/cybersecurity-data-pipeline"
sys.path.insert(0, repo_root)

import importlib
for mod in ['src.ingestion', 'src.cleaning', 'src.transformation', 'src.ml.train']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from src.ml.train import (
    prepare_training_data,
    train_random_forest,
    train_xgboost,
    evaluate_model,
    get_feature_importance,
    train_models,
    save_model_to_volume,
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("j9-ml-training").getOrCreate()
spark.sql("USE CATALOG workspace")

# Verify versions are compatible
import numpy, sklearn, xgboost
print(f"✓ numpy: {numpy.__version__}")
print(f"✓ scikit-learn: {sklearn.__version__}")
print(f"✓ xgboost: {xgboost.__version__}")
print(f"✓ Spark: {spark.version}")
print(f"✓ Catalog: workspace")
print(f"✓ ML modules imported")

In [0]:
# === Verify Gold table is ready for ML ===
print("--- Gold table summary ---")
spark.sql("SELECT COUNT(*) AS total_rows FROM workspace.cybersecurity.gold").show()

print("--- is_attack distribution ---")
spark.sql("""
    SELECT is_attack, COUNT(*) AS count,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM workspace.cybersecurity.gold), 2) AS percentage
    FROM workspace.cybersecurity.gold
    GROUP BY is_attack
    ORDER BY is_attack
""").show()

print("--- Column count ---")
gold_cols = spark.sql("SELECT * FROM workspace.cybersecurity.gold LIMIT 1").columns
print(f"  Total columns: {len(gold_cols)}")

In [0]:
# === Run the complete ML training pipeline ===
print("=" * 60)
print("Starting ML Training Pipeline")
print("=" * 60)

results = train_models(spark, sample_size=200000, random_state=42)

print("\n" + "=" * 60)
print(f"✓ Training completed!")
print(f"  Best model: {results['best_model_name']}")
print(f"  Best F1-score: {results['best_metrics']['f1_score']:.4f}")
print(f"  Best ROC AUC: {results['best_metrics']['roc_auc']:.4f}")
print(f"  Model saved at: {results['model_path']}")
print("=" * 60)

In [0]:
# === Detailed metrics inspection ===
import pandas as pd

print("=" * 60)
print("Models Comparison")
print("=" * 60)

comparison_df = pd.DataFrame([
    {
        'Model': 'Random Forest',
        'Accuracy': results['rf_metrics']['accuracy'],
        'Precision': results['rf_metrics']['precision'],
        'Recall': results['rf_metrics']['recall'],
        'F1-score': results['rf_metrics']['f1_score'],
        'ROC AUC': results['rf_metrics']['roc_auc'],
    },
    {
        'Model': 'XGBoost',
        'Accuracy': results['xgb_metrics']['accuracy'],
        'Precision': results['xgb_metrics']['precision'],
        'Recall': results['xgb_metrics']['recall'],
        'F1-score': results['xgb_metrics']['f1_score'],
        'ROC AUC': results['xgb_metrics']['roc_auc'],
    },
])
display(comparison_df)

print("\n--- Random Forest Confusion Matrix ---")
print(pd.DataFrame(
    results['rf_metrics']['confusion_matrix'],
    columns=['Predicted Benign', 'Predicted Attack'],
    index=['Actual Benign', 'Actual Attack']
))

print("\n--- XGBoost Confusion Matrix ---")
print(pd.DataFrame(
    results['xgb_metrics']['confusion_matrix'],
    columns=['Predicted Benign', 'Predicted Attack'],
    index=['Actual Benign', 'Actual Attack']
))

In [0]:
# === Feature importance comparison ===
print("=" * 60)
print("Top 15 Features - Random Forest")
print("=" * 60)
display(results['rf_importance'].head(15))

print("\n" + "=" * 60)
print("Top 15 Features - XGBoost")
print("=" * 60)
display(results['xgb_importance'].head(15))

# Common important features
rf_top10 = set(results['rf_importance'].head(10)['feature'])
xgb_top10 = set(results['xgb_importance'].head(10)['feature'])
common_important = rf_top10 & xgb_top10

print(f"\n--- Features important in BOTH models ({len(common_important)}) ---")
for feature in common_important:
    print(f"  - {feature}")

In [0]:
# === Verify the saved model can be loaded back ===
import joblib
import json

best_name = results['best_model_name']
model_path = results['model_path']

print(f"--- Loading saved model: {best_name} ---")
loaded_model = joblib.load(model_path)
print(f"✓ Model loaded: {type(loaded_model).__name__}")

# Load metrics
metrics_path = f"/Volumes/workspace/cybersecurity/models/{best_name}_metrics.json"
with open(metrics_path, 'r') as f:
    loaded_metrics = json.load(f)

print(f"\n--- Saved metrics ---")
for k, v in loaded_metrics.items():
    if isinstance(v, (int, float)):
        print(f"  {k}: {v:.4f}")
    elif k == 'confusion_matrix':
        print(f"  {k}: {v}")

# Load feature names
features_path = f"/Volumes/workspace/cybersecurity/models/{best_name}_features.json"
with open(features_path, 'r') as f:
    loaded_features = json.load(f)

print(f"\n--- Feature count ---")
print(f"  Total features: {len(loaded_features)}")
print(f"  First 5: {loaded_features[:5]}")